In [3]:
import duckdb
import pandas as pd

sales = pd.DataFrame({
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1009, 1010
    ],
    "customer_id": [
        "C1", "C2", "C1", "C3", "C2",
        "C1", "C3", "C2", "C1", "C3"
    ],
    "order_date": pd.to_datetime([
        "2026-01-05",
        "2026-01-18",
        "2026-02-03",
        "2026-02-14",
        "2026-02-20",
        "2026-03-01",
        "2026-03-10",
        "2026-03-15",
        "2026-03-22",
        "2026-03-28"
    ]),
    "amount": [
        120.0,
        340.5,
        89.0,
        210.0,
        55.5,
        430.0,
        199.0,
        310.0,
        75.0,
        265.0
    ]
})

customers = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3"],
    "region": ["Ege", "Marmara", "İç Anadolu"]
})

duckdb.sql("""
SELECT *
FROM sales
ORDER BY order_id
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┐
│ order_id │ customer_id │     order_date      │ amount │
│  int64   │   varchar   │      timestamp      │ double │
├──────────┼─────────────┼─────────────────────┼────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │
│     1004 │ C3          │ 2026-02-14 00:00:00 │  210.0 │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │
│     1007 │ C3          │ 2026-03-10 00:00:00 │  199.0 │
│     1008 │ C2          │ 2026-03-15 00:00:00 │  310.0 │
│     1009 │ C1          │ 2026-03-22 00:00:00 │   75.0 │
│     1010 │ C3          │ 2026-03-28 00:00:00 │  265.0 │
└──────────┴─────────────┴─────────────────────┴────────┘
  10 rows                                     4 columns



In [4]:
duckdb.sql("""SELECT customer_id, SUM(amount) AS total_amount
FROM sales
GROUP BY customer_id
ORDER BY customer_id
""").show()

┌─────────────┬──────────────┐
│ customer_id │ total_amount │
│   varchar   │    double    │
├─────────────┼──────────────┤
│ C1          │        714.0 │
│ C2          │        706.0 │
│ C3          │        674.0 │
└─────────────┴──────────────┘



In [10]:
duckdb.sql("""
SELECT order_id,customer_id, amount,
    SUM(amount) OVER ( PARTITION BY customer_id) AS customer_total
FROM sales
ORDER BY customer_id, order_id
""").show()

┌──────────┬─────────────┬────────┬────────────────┐
│ order_id │ customer_id │ amount │ customer_total │
│  int64   │   varchar   │ double │     double     │
├──────────┼─────────────┼────────┼────────────────┤
│     1001 │ C1          │  120.0 │          714.0 │
│     1003 │ C1          │   89.0 │          714.0 │
│     1006 │ C1          │  430.0 │          714.0 │
│     1009 │ C1          │   75.0 │          714.0 │
│     1002 │ C2          │  340.5 │          706.0 │
│     1005 │ C2          │   55.5 │          706.0 │
│     1008 │ C2          │  310.0 │          706.0 │
│     1004 │ C3          │  210.0 │          674.0 │
│     1007 │ C3          │  199.0 │          674.0 │
│     1010 │ C3          │  265.0 │          674.0 │
└──────────┴─────────────┴────────┴────────────────┘
  10 rows                                4 columns



In [12]:
duckdb.sql("""
SELECT order_id,customer_id,amount,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS amount_rank
FROM sales
ORDER BY customer_id, amount_rank
""").show()

┌──────────┬─────────────┬────────┬─────────────┐
│ order_id │ customer_id │ amount │ amount_rank │
│  int64   │   varchar   │ double │    int64    │
├──────────┼─────────────┼────────┼─────────────┤
│     1006 │ C1          │  430.0 │           1 │
│     1001 │ C1          │  120.0 │           2 │
│     1003 │ C1          │   89.0 │           3 │
│     1009 │ C1          │   75.0 │           4 │
│     1002 │ C2          │  340.5 │           1 │
│     1008 │ C2          │  310.0 │           2 │
│     1005 │ C2          │   55.5 │           3 │
│     1010 │ C3          │  265.0 │           1 │
│     1004 │ C3          │  210.0 │           2 │
│     1007 │ C3          │  199.0 │           3 │
└──────────┴─────────────┴────────┴─────────────┘
  10 rows                             4 columns



In [13]:
ranking_test = pd.DataFrame({
    "name": ["A", "B", "C", "D", "E"],
    "score": [300, 300, 200, 100, 100]
})

In [14]:
duckdb.sql("""
SELECT name,score,
    ROW_NUMBER() OVER (ORDER BY score DESC) AS row_number_result,
    RANK() OVER (ORDER BY score DESC) AS rank_result,
    DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank_result
FROM ranking_test
ORDER BY score DESC, name
""").show()

┌─────────┬───────┬───────────────────┬─────────────┬───────────────────┐
│  name   │ score │ row_number_result │ rank_result │ dense_rank_result │
│ varchar │ int64 │       int64       │    int64    │       int64       │
├─────────┼───────┼───────────────────┼─────────────┼───────────────────┤
│ A       │   300 │                 1 │           1 │                 1 │
│ B       │   300 │                 2 │           1 │                 1 │
│ C       │   200 │                 3 │           3 │                 2 │
│ D       │   100 │                 4 │           4 │                 3 │
│ E       │   100 │                 5 │           4 │                 3 │
└─────────┴───────┴───────────────────┴─────────────┴───────────────────┘



In [19]:
duckdb.sql("""
SELECT name, score,
DENSE_RANK() OVER (ORDER BY score DESC) AS score_level
FROM ranking_test
QUALIFY score_level <= 2
ORDER BY score DESC, name
""").show()

┌─────────┬───────┬─────────────┐
│  name   │ score │ score_level │
│ varchar │ int64 │    int64    │
├─────────┼───────┼─────────────┤
│ A       │   300 │           1 │
│ B       │   300 │           1 │
│ C       │   200 │           2 │
└─────────┴───────┴─────────────┘



In [24]:
duckdb.sql("""
SELECT order_id,customer_id,order_date,amount,
LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_amount,
amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS amount_difference
FROM sales
ORDER BY customer_id, order_date
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬─────────────────┬───────────────────┐
│ order_id │ customer_id │     order_date      │ amount │ previous_amount │ amount_difference │
│  int64   │   varchar   │      timestamp      │ double │     double      │      double       │
├──────────┼─────────────┼─────────────────────┼────────┼─────────────────┼───────────────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │            NULL │              NULL │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │           120.0 │             -31.0 │
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │            89.0 │             341.0 │
│     1009 │ C1          │ 2026-03-22 00:00:00 │   75.0 │           430.0 │            -355.0 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │            NULL │              NULL │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │           340.5 │            -285.0 │
│     1008 │ C2          │ 2026-03-15 00

In [25]:
duckdb.sql("""
SELECT order_id,customer_id,order_date,amount,
LEAD(amount) OVER (PARTITION BY customer_id ORDER BY order_date ) AS next_amount,
LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date ) - order_date AS time_until_next_order
FROM sales
ORDER BY customer_id, order_date
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬─────────────┬───────────────────────┐
│ order_id │ customer_id │     order_date      │ amount │ next_amount │ time_until_next_order │
│  int64   │   varchar   │      timestamp      │ double │   double    │       interval        │
├──────────┼─────────────┼─────────────────────┼────────┼─────────────┼───────────────────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │        89.0 │ 29 days               │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │       430.0 │ 26 days               │
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │        75.0 │ 21 days               │
│     1009 │ C1          │ 2026-03-22 00:00:00 │   75.0 │        NULL │ NULL                  │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │        55.5 │ 33 days               │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │       310.0 │ 23 days               │
│     1008 │ C2          │ 2026-03-15 00

In [26]:
duckdb.sql("""
SELECT order_id,customer_id,order_date,amount,
SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM sales
ORDER BY customer_id, order_date
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬───────────────┐
│ order_id │ customer_id │     order_date      │ amount │ running_total │
│  int64   │   varchar   │      timestamp      │ double │    double     │
├──────────┼─────────────┼─────────────────────┼────────┼───────────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │         120.0 │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │         209.0 │
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │         639.0 │
│     1009 │ C1          │ 2026-03-22 00:00:00 │   75.0 │         714.0 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │         340.5 │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │         396.0 │
│     1008 │ C2          │ 2026-03-15 00:00:00 │  310.0 │         706.0 │
│     1004 │ C3          │ 2026-02-14 00:00:00 │  210.0 │         210.0 │
│     1007 │ C3          │ 2026-03-10 00:00:00 │  199.0 │         409.0 │
│     1010 │ C3          │ 2026-03-28 

In [27]:
duckdb.sql("""
SELECT order_id,customer_id,order_date,amount,
AVG(amount) OVER (PARTITION BY customer_id ORDER BY order_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_average_3
FROM sales
ORDER BY customer_id, order_date
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬────────────────────┐
│ order_id │ customer_id │     order_date      │ amount │  moving_average_3  │
│  int64   │   varchar   │      timestamp      │ double │       double       │
├──────────┼─────────────┼─────────────────────┼────────┼────────────────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │              120.0 │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │              104.5 │
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │              213.0 │
│     1009 │ C1          │ 2026-03-22 00:00:00 │   75.0 │              198.0 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │              340.5 │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │              198.0 │
│     1008 │ C2          │ 2026-03-15 00:00:00 │  310.0 │ 235.33333333333334 │
│     1004 │ C3          │ 2026-02-14 00:00:00 │  210.0 │              210.0 │
│     1007 │ C3          │ 2026-03-10 00:00:00 │  19

In [28]:
duckdb.sql("""
SELECT order_id,customer_id,order_date,amount,
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS amount_rank
FROM sales
QUALIFY amount_rank = 1
ORDER BY customer_id
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬─────────────┐
│ order_id │ customer_id │     order_date      │ amount │ amount_rank │
│  int64   │   varchar   │      timestamp      │ double │    int64    │
├──────────┼─────────────┼─────────────────────┼────────┼─────────────┤
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │           1 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │           1 │
│     1010 │ C3          │ 2026-03-28 00:00:00 │  265.0 │           1 │
└──────────┴─────────────┴─────────────────────┴────────┴─────────────┘



In [29]:
duckdb.sql("""
WITH ranked_sales AS (
    SELECT order_id,customer_id,order_date,amount,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS amount_rank
    FROM sales
)
SELECT *
FROM ranked_sales
WHERE amount_rank = 1
ORDER BY customer_id
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬─────────────┐
│ order_id │ customer_id │     order_date      │ amount │ amount_rank │
│  int64   │   varchar   │      timestamp      │ double │    int64    │
├──────────┼─────────────┼─────────────────────┼────────┼─────────────┤
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │           1 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │           1 │
│     1010 │ C3          │ 2026-03-28 00:00:00 │  265.0 │           1 │
└──────────┴─────────────┴─────────────────────┴────────┴─────────────┘



In [30]:
duckdb.sql("""
SELECT order_id, order_date, DATE_TRUNC('month', order_date) AS order_month,amount
FROM sales
ORDER BY order_date
""").show()

┌──────────┬─────────────────────┬─────────────────────┬────────┐
│ order_id │     order_date      │     order_month     │ amount │
│  int64   │      timestamp      │      timestamp      │ double │
├──────────┼─────────────────────┼─────────────────────┼────────┤
│     1001 │ 2026-01-05 00:00:00 │ 2026-01-01 00:00:00 │  120.0 │
│     1002 │ 2026-01-18 00:00:00 │ 2026-01-01 00:00:00 │  340.5 │
│     1003 │ 2026-02-03 00:00:00 │ 2026-02-01 00:00:00 │   89.0 │
│     1004 │ 2026-02-14 00:00:00 │ 2026-02-01 00:00:00 │  210.0 │
│     1005 │ 2026-02-20 00:00:00 │ 2026-02-01 00:00:00 │   55.5 │
│     1006 │ 2026-03-01 00:00:00 │ 2026-03-01 00:00:00 │  430.0 │
│     1007 │ 2026-03-10 00:00:00 │ 2026-03-01 00:00:00 │  199.0 │
│     1008 │ 2026-03-15 00:00:00 │ 2026-03-01 00:00:00 │  310.0 │
│     1009 │ 2026-03-22 00:00:00 │ 2026-03-01 00:00:00 │   75.0 │
│     1010 │ 2026-03-28 00:00:00 │ 2026-03-01 00:00:00 │  265.0 │
└──────────┴─────────────────────┴─────────────────────┴────────┘
  10 rows 

In [36]:
import duckdb
import pandas as pd

# S8'deki sales tablosuna benzer, + gruplama için customer_id eklendi
sales = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "customer_id": ["C1", "C2", "C1", "C3", "C2", "C1", "C3", "C2", "C1", "C3"],
    "order_date": pd.to_datetime([
        "2026-01-05", "2026-01-18", "2026-02-03", "2026-02-14", "2026-02-20",
        "2026-03-01", "2026-03-10", "2026-03-15", "2026-03-22", "2026-03-28"
    ]),
    "amount": [120.0, 340.5, 89.0, 210.0, 55.5, 430.0, 199.0, 310.0, 75.0, 265.0]
})

customers = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3"],
    "region": ["Ege", "Marmara", "İç Anadolu"]
})

# DuckDB'ye DOSYA'ya gerek kalmadan doğrudan pandas df sorgulanabilir:
duckdb.sql("SELECT * FROM sales LIMIT 5").show()

┌──────────┬─────────────┬─────────────────────┬────────┐
│ order_id │ customer_id │     order_date      │ amount │
│  int64   │   varchar   │      timestamp      │ double │
├──────────┼─────────────┼─────────────────────┼────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │
│     1004 │ C3          │ 2026-02-14 00:00:00 │  210.0 │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │
└──────────┴─────────────┴─────────────────────┴────────┘



01 · Customer-Level Aggregation

Calculate the following statistics for each customer:

Total order amount
Average order amount<br>
Expected columns: 
customer_id	
total_amount	
average_amount
<details> <summary><b>Concepts</b></summary>

GROUP BY, SUM, AVG

</details>

In [40]:
duckdb.sql("""
SELECT customer_id, SUM(amount) ,AVG(amount) FROM sales GROUP BY customer_id 
""").show()

┌─────────────┬─────────────┬────────────────────┐
│ customer_id │ sum(amount) │    avg(amount)     │
│   varchar   │   double    │       double       │
├─────────────┼─────────────┼────────────────────┤
│ C3          │       674.0 │ 224.66666666666666 │
│ C2          │       706.0 │ 235.33333333333334 │
│ C1          │       714.0 │              178.5 │
└─────────────┴─────────────┴────────────────────┘



02 · Rank Orders Within Each Customer

Rank every customer's orders from the highest amount to the lowest.

The largest order of each customer must receive rank 1.

Expected columns:
Column	
order_id	
customer_id
amount	
amount_rank	
<details> <summary><b>Concepts</b></summary>

ROW_NUMBER, OVER, PARTITION BY, window ORDER BY

</details>

In [52]:
duckdb.sql("""
SELECT order_id, customer_id, amount, 
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) as amount_rank
FROM sales
ORDER BY customer_id , amount_rank
""").show()

┌──────────┬─────────────┬────────┬─────────────┐
│ order_id │ customer_id │ amount │ amount_rank │
│  int64   │   varchar   │ double │    int64    │
├──────────┼─────────────┼────────┼─────────────┤
│     1006 │ C1          │  430.0 │           1 │
│     1001 │ C1          │  120.0 │           2 │
│     1003 │ C1          │   89.0 │           3 │
│     1009 │ C1          │   75.0 │           4 │
│     1002 │ C2          │  340.5 │           1 │
│     1008 │ C2          │  310.0 │           2 │
│     1005 │ C2          │   55.5 │           3 │
│     1010 │ C3          │  265.0 │           1 │
│     1004 │ C3          │  210.0 │           2 │
│     1007 │ C3          │  199.0 │           3 │
└──────────┴─────────────┴────────┴─────────────┘
  10 rows                             4 columns



03 · Compare Each Order with the Previous Order

For each customer, compare every order with their previous order.

Orders must be processed chronologically.

Calculate:

amount difference=current amount−previous amount
Expected columns:
order_id
customer_id	
order_date
amount
previous_amount	
amount_difference<br>

The first order of each customer should return NULL for both
previous_amount and amount_difference.

<details> <summary><b>Concepts</b></summary>

LAG, PARTITION BY, chronological window ordering

</details>

In [43]:
duckdb.sql("""
SELECT order_id,customer_id,order_date,amount,
amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS amount_difference
FROM sales
ORDER BY customer_id, order_date
""").show()

┌──────────┬─────────────┬─────────────────────┬────────┬───────────────────┐
│ order_id │ customer_id │     order_date      │ amount │ amount_difference │
│  int64   │   varchar   │      timestamp      │ double │      double       │
├──────────┼─────────────┼─────────────────────┼────────┼───────────────────┤
│     1001 │ C1          │ 2026-01-05 00:00:00 │  120.0 │              NULL │
│     1003 │ C1          │ 2026-02-03 00:00:00 │   89.0 │             -31.0 │
│     1006 │ C1          │ 2026-03-01 00:00:00 │  430.0 │             341.0 │
│     1009 │ C1          │ 2026-03-22 00:00:00 │   75.0 │            -355.0 │
│     1002 │ C2          │ 2026-01-18 00:00:00 │  340.5 │              NULL │
│     1005 │ C2          │ 2026-02-20 00:00:00 │   55.5 │            -285.0 │
│     1008 │ C2          │ 2026-03-15 00:00:00 │  310.0 │             254.5 │
│     1004 │ C3          │ 2026-02-14 00:00:00 │  210.0 │              NULL │
│     1007 │ C3          │ 2026-03-10 00:00:00 │  199.0 │       

04 · Add Customer Region Information

Join the sales DataFrame with the customers DataFrame using customer_id.

Add the corresponding region to every order.

Expected columns:
order_id	
customer_id	
region	
order_date	
amount<br>

Sort the final result by order_id.

<details> <summary><b>Concepts</b></summary>

JOIN, table aliases, join conditions

</details>

In [44]:
duckdb.sql("""
SELECT s.order_id, s.customer_id,c.region,s.order_date,s.amount
FROM sales AS s
JOIN customers AS c
    ON s.customer_id = c.customer_id
ORDER BY s.order_id
""").show()

┌──────────┬─────────────┬────────────┬─────────────────────┬────────┐
│ order_id │ customer_id │   region   │     order_date      │ amount │
│  int64   │   varchar   │  varchar   │      timestamp      │ double │
├──────────┼─────────────┼────────────┼─────────────────────┼────────┤
│     1001 │ C1          │ Ege        │ 2026-01-05 00:00:00 │  120.0 │
│     1002 │ C2          │ Marmara    │ 2026-01-18 00:00:00 │  340.5 │
│     1003 │ C1          │ Ege        │ 2026-02-03 00:00:00 │   89.0 │
│     1004 │ C3          │ İç Anadolu │ 2026-02-14 00:00:00 │  210.0 │
│     1005 │ C2          │ Marmara    │ 2026-02-20 00:00:00 │   55.5 │
│     1006 │ C1          │ Ege        │ 2026-03-01 00:00:00 │  430.0 │
│     1007 │ C3          │ İç Anadolu │ 2026-03-10 00:00:00 │  199.0 │
│     1008 │ C2          │ Marmara    │ 2026-03-15 00:00:00 │  310.0 │
│     1009 │ C1          │ Ege        │ 2026-03-22 00:00:00 │   75.0 │
│     1010 │ C3          │ İç Anadolu │ 2026-03-28 00:00:00 │  265.0 │
└─────

05 · Monthly Sales Aggregation

Calculate the total sales amount for each month.

Use:

DATE_TRUNC('month', order_date)

to convert every order date to the beginning of its month.

Expected columns:
order_month	
total_amount

Sort the result chronologically.

<details> <summary><b>Concepts</b></summary>

DATE_TRUNC, GROUP BY, monthly aggregation

</details>

In [53]:
duckdb.sql("""
SELECT DATE_TRUNC('month',order_date) as order_month, SUM(amount) as total_amount
FROM sales
GROUP BY order_month
ORDER BY order_month
""").show()

┌─────────────────────┬──────────────┐
│     order_month     │ total_amount │
│      timestamp      │    double    │
├─────────────────────┼──────────────┤
│ 2026-01-01 00:00:00 │        460.5 │
│ 2026-02-01 00:00:00 │        354.5 │
│ 2026-03-01 00:00:00 │       1279.0 │
└─────────────────────┴──────────────┘

